
# Offline GEO-filtered RAG over PDFs with LangChain + FAISS + Hugging Face + Qwen2.5-3B-Instruct

This notebook builds a local/offline RAG pipeline that:

- reads PDFs from `APAC`, `EMEA`, and `AMER` folders
- attaches GEO metadata to every page and chunk
- splits documents into chunks
- creates embeddings with a **local Hugging Face embedding model**
- builds **three separate FAISS indexes** (one per GEO)
- supports queries over **one GEO or multiple GEOs** like `APAC and AMER`
- retrieves only from the requested GEOs when the query mentions them
- uses **Qwen/Qwen2.5-3B-Instruct** as the **local/offline LLM**
- returns **"I do not have the answer in the provided documents."** when context is insufficient

## Why this model

`Qwen/Qwen2.5-3B-Instruct` is a much lighter local instruction model than `openai/gpt-oss-20b`,
and its model card documents standard Transformers usage and a long context window.
This makes it a more practical choice for a notebook-based local RAG prototype.


In [1]:

# Install the core packages if needed
# Restart the kernel after installation if this is your first run.

# %pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers transformers accelerate torch tqdm



## Expected data layout

```text
data/
├── APAC/
│   ├── file1.pdf
│   └── file2.pdf
├── EMEA/
│   └── file3.pdf
└── AMER/
    └── file4.pdf
```

## Notes

- `PyPDFLoader` is best for text PDFs or PDFs that already contain a text layer.
- OCR fallback can be added later for scanned/image-only PDFs.
- The embedding model is separate from the chat model.
- This notebook uses the Transformers chat pipeline with message-style input.


In [2]:

from __future__ import annotations

import hashlib
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from tqdm.auto import tqdm

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import pipeline


In [3]:

@dataclass
class Settings:
    data_dir: Path = Path("data")
    storage_dir: Path = Path("storage")
    geos: Tuple[str, ...] = ("APAC", "EMEA", "AMER")

    # Chunking
    chunk_size: int = 800
    chunk_overlap: int = 100

    # Retrieval
    top_k: int = 3

    # Local embedding model
    embedding_model: str = "sentence-transformers/all-mpnet-base-v2"

    # Local instruction model
    llm_model: str = "Qwen/Qwen2.5-3B-Instruct"

    # Generation
    max_new_tokens: int = 128
    do_sample: bool = False
    temperature: float = 0.0

    # Optional text quality guard
    min_page_chars: int = 40

    # Context trimming to keep memory use lower
    max_chars_per_chunk_in_prompt: int = 1200

settings = Settings()
settings.storage_dir.mkdir(parents=True, exist_ok=True)

settings


Settings(data_dir=WindowsPath('data'), storage_dir=WindowsPath('storage'), geos=('APAC', 'EMEA', 'AMER'), chunk_size=800, chunk_overlap=100, top_k=3, embedding_model='sentence-transformers/all-mpnet-base-v2', llm_model='Qwen/Qwen2.5-3B-Instruct', max_new_tokens=128, do_sample=False, temperature=0.0, min_page_chars=40, max_chars_per_chunk_in_prompt=1200)

## Helper functions

In [4]:

def normalize_geo(value: str) -> str:
    value = value.strip().upper()
    if value not in settings.geos:
        raise ValueError(f"Unsupported GEO: {value}")
    return value


def detect_geos_from_query(query: str) -> List[str]:
    text = query.upper()
    found = []

    for geo in settings.geos:
        if re.search(rf"\b{geo}\b", text):
            found.append(geo)

    return found


def make_doc_id(path: Path) -> str:
    return hashlib.md5(str(path.resolve()).encode("utf-8")).hexdigest()


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def page_has_usable_text(text: str, min_chars: int = settings.min_page_chars) -> bool:
    text = clean_text(text)
    return len(text) >= min_chars


def build_chunk_id(source: str, page: Optional[int], idx: int) -> str:
    raw = f"{source}|{page}|{idx}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


## 1) Load PDFs and attach metadata

In [5]:

def load_geo_documents(data_dir: Path, geo: str) -> List[Document]:
    geo = normalize_geo(geo)
    geo_dir = data_dir / geo
    if not geo_dir.exists():
        print(f"Warning: {geo_dir} does not exist")
        return []

    all_docs: List[Document] = []
    pdf_paths = sorted(geo_dir.rglob("*.pdf"))

    for pdf_path in tqdm(pdf_paths, desc=f"Loading {geo} PDFs"):
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()

        doc_id = make_doc_id(pdf_path)

        for page_doc in pages:
            raw_text = page_doc.page_content or ""
            cleaned = clean_text(raw_text)

            if not page_has_usable_text(cleaned):
                # Later you can replace this with OCR fallback.
                continue

            metadata = dict(page_doc.metadata)
            metadata.update(
                {
                    "geo": geo,
                    "source": str(pdf_path),
                    "doc_id": doc_id,
                    "file_name": pdf_path.name,
                    "page": metadata.get("page"),
                }
            )

            all_docs.append(
                Document(
                    page_content=cleaned,
                    metadata=metadata,
                )
            )

    return all_docs


In [6]:

geo_page_docs: Dict[str, List[Document]] = {}

for geo in settings.geos:
    docs = load_geo_documents(settings.data_dir, geo)
    geo_page_docs[geo] = docs
    print(f"{geo}: {len(docs)} usable page documents")


Loading APAC PDFs:   0%|          | 0/2 [00:00<?, ?it/s]

APAC: 116 usable page documents


Loading EMEA PDFs:   0%|          | 0/2 [00:00<?, ?it/s]

EMEA: 79 usable page documents


Loading AMER PDFs:   0%|          | 0/1 [00:00<?, ?it/s]

AMER: 17 usable page documents


## 2) Split into chunks

In [7]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)


def chunk_documents(docs: List[Document]) -> List[Document]:
    chunks = text_splitter.split_documents(docs)

    for idx, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "")
        page = chunk.metadata.get("page")
        chunk.metadata["chunk_id"] = build_chunk_id(source, page, idx)

    return chunks


In [8]:

geo_chunk_docs: Dict[str, List[Document]] = {}

for geo in settings.geos:
    chunks = chunk_documents(geo_page_docs[geo])
    geo_chunk_docs[geo] = chunks
    print(f"{geo}: {len(chunks)} chunks")


APAC: 157 chunks
EMEA: 161 chunks
AMER: 43 chunks


## 3) Create local embeddings

In [9]:

embeddings = HuggingFaceEmbeddings(model_name=settings.embedding_model)
embeddings


HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## 4) Build and save one FAISS index per GEO

In [10]:

def geo_index_path(geo: str) -> Path:
    return settings.storage_dir / f"faiss_{geo.lower()}"


def build_and_save_geo_indexes(
    geo_to_chunks: Dict[str, List[Document]],
    embeddings_model: HuggingFaceEmbeddings,
) -> Dict[str, FAISS]:
    stores: Dict[str, FAISS] = {}

    for geo, chunks in geo_to_chunks.items():
        if not chunks:
            print(f"Skipping {geo}: no chunks found")
            continue

        print(f"Building FAISS index for {geo} with {len(chunks)} chunks...")
        vectorstore = FAISS.from_documents(chunks, embeddings_model)

        save_dir = geo_index_path(geo)
        save_dir.mkdir(parents=True, exist_ok=True)
        vectorstore.save_local(str(save_dir))

        stores[geo] = vectorstore

    return stores


In [11]:

# Run this once to build the indexes
geo_vectorstores = build_and_save_geo_indexes(geo_chunk_docs, embeddings)


Building FAISS index for APAC with 157 chunks...
Building FAISS index for EMEA with 161 chunks...
Building FAISS index for AMER with 43 chunks...


## 5) Load saved FAISS indexes

In [12]:

def load_geo_indexes(embeddings_model: HuggingFaceEmbeddings) -> Dict[str, FAISS]:
    stores: Dict[str, FAISS] = {}

    for geo in settings.geos:
        save_dir = geo_index_path(geo)
        if save_dir.exists():
            stores[geo] = FAISS.load_local(
                str(save_dir),
                embeddings_model,
                allow_dangerous_deserialization=True,
            )
    return stores


# Uncomment in a fresh session after indexes already exist:
# geo_vectorstores = load_geo_indexes(embeddings)


## 6) Load the local instruction model

In [13]:

generator = pipeline(
    "text-generation",
    model=settings.llm_model,
    torch_dtype="auto",
    device_map="auto",
)

generator


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0



## 7) Retrieval and answer generation

This version supports:

- single GEO: `APAC`
- multiple GEOs: `APAC and AMER`
- no GEO mentioned: searches all available GEO indexes


In [14]:

def format_context(docs: List[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs[: settings.top_k], start=1):
        source = doc.metadata.get("file_name", "unknown")
        page = doc.metadata.get("page", "unknown")
        geo = doc.metadata.get("geo", "unknown")
        text = doc.page_content[: settings.max_chars_per_chunk_in_prompt]
        blocks.append(
            f"[Chunk {i}]\n"
            f"GEO: {geo}\n"
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content:\n{text}"
        )
    return "\n\n---\n\n".join(blocks)


def retrieve_documents(
    question: str,
    stores: Dict[str, FAISS],
    k: int = settings.top_k,
) -> Tuple[List[str], List[Document]]:
    requested_geos = detect_geos_from_query(question)
    target_geos = requested_geos if requested_geos else list(stores.keys())

    scored_results = []

    for geo in target_geos:
        if geo not in stores:
            continue

        docs_and_scores = stores[geo].similarity_search_with_score(question, k=k)
        for doc, score in docs_and_scores:
            scored_results.append((doc, score))

    scored_results.sort(key=lambda x: x[1])  # lower score is better

    final_docs = []
    seen_keys = set()

    for doc, score in scored_results:
        key = (
            doc.metadata.get("file_name"),
            doc.metadata.get("page"),
            doc.metadata.get("chunk_id"),
        )
        if key not in seen_keys:
            seen_keys.add(key)
            final_docs.append(doc)

        if len(final_docs) >= k:
            break

    return target_geos, final_docs


In [15]:

SYSTEM_INSTRUCTION = (
    "You are a retrieval-augmented assistant. "
    "Answer only from the supplied context. "
    "Do not use outside knowledge. "
    "If the answer is not present in the context, reply exactly with: "
    "'I do not have the answer in the provided documents.'"
)


def extract_assistant_text(generated_output) -> str:
    generated = generated_output[0]["generated_text"]

    if isinstance(generated, list):
        last_item = generated[-1]
        if isinstance(last_item, dict):
            content = last_item.get("content", "")
            if isinstance(content, str):
                return content.strip()

    return str(generated).strip()


def answer_question(question: str, stores: Dict[str, FAISS]) -> Dict[str, object]:
    geos, docs = retrieve_documents(question, stores)

    if not docs:
        return {
            "question": question,
            "geo_used": geos,
            "answer": "I do not have the answer in the provided documents.",
            "sources": [],
        }

    context = format_context(docs)
    geo_label = ", ".join(geos) if geos else "ALL"

    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": (
                f"GEO filter: {geo_label}\n\n"
                f"Context:\n{context}\n\n"
                f"Question: {question}"
            ),
        },
    ]

    outputs = generator(
        messages,
        max_new_tokens=settings.max_new_tokens,
        do_sample=settings.do_sample,
        temperature=settings.temperature,
    )

    answer = extract_assistant_text(outputs)

    sources = [
        {
            "geo": doc.metadata.get("geo"),
            "file_name": doc.metadata.get("file_name"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
        }
        for doc in docs
    ]

    return {
        "question": question,
        "geo_used": geos,
        "answer": answer,
        "sources": sources,
    }


## 8) Example queries

In [26]:

result = answer_question("Transformer mentioned?  EMEA", geo_vectorstores)
print(result["answer"])
print(json.dumps(result["sources"][:3], indent=2))


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The transformer mentioned in the context is a Three Phase transformer, specifically designed for use in high-voltage alternating-current circuits. It is described as being used in conjunction with standard low-range AC instruments to measure currents and voltages in high-voltage circuits where direct connection of instruments would be impractical.
[
  {
    "geo": "EMEA",
    "file_name": "092ea5dd-f471-4578-b6cb-9fcb17a9b84a.pdf",
    "page": 28,
    "chunk_id": "22e5bc6429730f85aaaf14d7dd25e828"
  },
  {
    "geo": "EMEA",
    "file_name": "092ea5dd-f471-4578-b6cb-9fcb17a9b84a.pdf",
    "page": 1,
    "chunk_id": "ff94a52acc0e516a80d8c41108421b5f"
  },
  {
    "geo": "EMEA",
    "file_name": "092ea5dd-f471-4578-b6cb-9fcb17a9b84a.pdf",
    "page": 16,
    "chunk_id": "8b6fada02c7cc88aaa26fe60edd23c99"
  }
]


In [20]:

result = answer_question("Compare the Transformer policy for APAC and AMER", geo_vectorstores)
print(result["answer"])
print(json.dumps(result["sources"][:5], indent=2))


I do not have the answer in the provided documents.
[
  {
    "geo": "APAC",
    "file_name": "2.pdf",
    "page": 88,
    "chunk_id": "247fd7fc3372e9d004c371830567b277"
  },
  {
    "geo": "APAC",
    "file_name": "2.pdf",
    "page": 92,
    "chunk_id": "1da06a02c63b0b560792e4a6a09b4544"
  },
  {
    "geo": "APAC",
    "file_name": "2.pdf",
    "page": 90,
    "chunk_id": "1fe03b76882252500f87acd0fb151960"
  }
]


In [21]:

result = answer_question("What are the Transformer ?", geo_vectorstores)
print(result["answer"])
print(json.dumps(result["sources"][:3], indent=2))


Transformers are devices that transfer electrical energy from one circuit to another through electromagnetic induction. They consist of two inductive windings on a core, which may be insulated from each other and the core. The core is typically made up of stacked laminated sheets of steel to minimize eddy currents and ensure a continuous magnetic path. Transformers can be found in various sizes and configurations depending on their power rating and application, ranging from small potential transformers with ratings of 40 to 100 W to larger units used in power distribution systems.
[
  {
    "geo": "EMEA",
    "file_name": "092ea5dd-f471-4578-b6cb-9fcb17a9b84a.pdf",
    "page": 29,
    "chunk_id": "919148f31e1448b94f5d173680721f1a"
  },
  {
    "geo": "AMER",
    "file_name": "11.pdf",
    "page": 0,
    "chunk_id": "03090b66adf26cd0765a64965ac4db0b"
  },
  {
    "geo": "APAC",
    "file_name": "2.pdf",
    "page": 88,
    "chunk_id": "247fd7fc3372e9d004c371830567b277"
  }
]



## 9) If memory is still tight

Try these settings:

```python
settings.top_k = 2
settings.max_new_tokens = 64
settings.max_chars_per_chunk_in_prompt = 800
```

If needed, you can also switch the model to `Qwen/Qwen2.5-1.5B-Instruct`.



## 10) Summary

This notebook keeps your original RAG design but adds multi-GEO retrieval:

- Loader: `PyPDFLoader`
- Splitter: `RecursiveCharacterTextSplitter`
- Embeddings: `HuggingFaceEmbeddings`
- Vector store: `FAISS`
- LLM: `Qwen/Qwen2.5-3B-Instruct`
- GEO filter: one FAISS index per GEO
- Query support: single GEO, multiple GEOs, or all GEOs
